# 🔬 Complete Appraisal PDF & Engagement Letter Generation System
**Full OCR + Detection → Appraisal Reports + Engagement Letters**

**Note:** Engagement letters and appraisal reports contain MATCHING data (same address, borrower, lender, etc.)

In [ ]:
# Install all dependencies
!pip install -q PyMuPDF pdf2image pytesseract opencv-python-headless reportlab Faker pandas pillow numpy tabulate
!apt-get install -qq tesseract-ocr poppler-utils
print("✅ Dependencies ready!")

In [ ]:
# Upload PDF
from google.colab import files
print("📤 Upload your appraisal PDF...")
uploaded = files.upload()
INPUT_PDF = list(uploaded.keys())[0]
print(f"✅ Loaded: {INPUT_PDF}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 1: COMPLETE PDF DETECTION ENGINE
# ═══════════════════════════════════════════════════════════════════
import fitz
import cv2
import numpy as np
from PIL import Image
import pytesseract
from pdf2image import convert_from_path
import json
import re
from dataclasses import dataclass
from typing import List, Dict, Any
from collections import defaultdict

@dataclass
class DetectedElement:
    element_type: str
    page: int
    x: int
    y: int
    w: int
    h: int
    content: Any = None
    is_dynamic: bool = True
    confidence: float = 0.0

class CompletePDFDetector:
    FIELD_PATTERNS = {
        'property_address': r'Property\s*Address[:\s]*([^\n]{5,50})',
        'city': r'City[:\s]*([A-Za-z\s]{2,30})',
        'state': r'State[:\s]*([A-Z]{2})',
        'zip': r'Zip\s*(?:Code)?[:\s]*(\d{5}(?:-\d{4})?)',
        'county': r'County[:\s]*([A-Za-z\s-]{3,40})',
        'borrower_name': r'Borrower[:\s]*([A-Za-z\s]{3,50})',
        'lender_name': r'Lender(?:\/Client)?[:\s]*([^\n]{5,60})',
        'appraiser_name': r'Appraiser[:\s]*([A-Za-z\s]{3,50})',
        'year_built': r'Year\s*Built[:\s]*(\d{4})',
        'gla': r'(?:Gross\s*Living\s*Area|GLA)[:\s]*([\d,]+)',
        'bedrooms': r'(?:Bedrooms|BR)[:\s]*(\d{1,2})',
        'bathrooms': r'(?:Bathrooms|Bath|BA)[:\s]*([\d.]+)',
        'appraised_value': r'(?:Appraised|Opinion\s*of|Market)\s*Value[:\s$]*([\d,]+)',
    }
    
    def __init__(self, pdf_path, dpi=150):
        self.pdf_path = pdf_path
        self.dpi = dpi
        self.doc = fitz.open(pdf_path)
        self.page_images = []
        self.elements: List[DetectedElement] = []
        self.master_data = {}
        self.page_layouts = []
        
    def detect_all(self):
        print(f"\n{'='*60}")
        print(f"🔬 COMPLETE PDF DETECTION - {len(self.doc)} pages")
        print(f"{'='*60}\n")
        self._render_pages()
        self._detect_text_blocks()
        self._extract_all_fields()
        self._print_detection_report()
        return self
    
    def _render_pages(self):
        print("📸 Rendering pages...")
        self.page_images = convert_from_path(self.pdf_path, dpi=self.dpi)
        for i, img in enumerate(self.page_images):
            self.page_layouts.append({'page': i+1, 'width': img.width, 'height': img.height, 'dpi': self.dpi})
        print(f"   ✅ Rendered {len(self.page_images)} pages")
    
    def _detect_text_blocks(self):
        print("📝 Detecting text blocks...")
        total = 0
        for i, img in enumerate(self.page_images):
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            ocr = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)
            for j in range(len(ocr['text'])):
                text = ocr['text'][j].strip()
                if text and int(ocr['conf'][j]) > 30:
                    self.elements.append(DetectedElement(
                        element_type='text', page=i+1,
                        x=ocr['left'][j], y=ocr['top'][j],
                        w=ocr['width'][j], h=ocr['height'][j],
                        content=text, confidence=int(ocr['conf'][j])
                    ))
                    total += 1
        print(f"   ✅ Found {total} text blocks")
    
    def _extract_all_fields(self):
        print("📋 Extracting fields...")
        all_text = " ".join([e.content for e in self.elements if e.element_type == 'text' and e.content])
        for field, pattern in self.FIELD_PATTERNS.items():
            match = re.search(pattern, all_text, re.IGNORECASE | re.DOTALL)
            if match:
                self.master_data[field] = match.group(1).strip()
        print(f"   ✅ Extracted {len(self.master_data)} fields")
    
    def _print_detection_report(self):
        print(f"\n📝 Extracted Fields ({len(self.master_data)}):")
        for k, v in list(self.master_data.items())[:10]:
            print(f"   {k:25} : {v[:40] if len(str(v)) > 40 else v}")

detector = CompletePDFDetector(INPUT_PDF)
detector.detect_all()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 2: UNIFIED DATA GENERATOR (SHARED BETWEEN BOTH DOCUMENTS)
# ═══════════════════════════════════════════════════════════════════
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

class UnifiedDataGenerator:
    """
    Generates UNIFIED data that is shared EXACTLY between:
    - Appraisal Report PDF
    - Engagement Letter PDF
    All matching fields are IDENTICAL word-for-word.
    """
    
    STATES = ['FL', 'CA', 'TX', 'NY', 'AZ', 'CO', 'GA', 'NC', 'VA', 'WA']
    PROPERTY_TYPES = ['Single Family Residential', 'Condominium', 'Townhouse', 'PUD']
    OCCUPANCY_TYPES = ['Owner Occupied', 'Tenant Occupied', 'Vacant']
    REPORT_TYPES = ['URAR', 'FHA', 'VA']
    
    def __init__(self):
        self.count = 0
    
    def generate(self) -> Dict:
        """Generate one complete dataset used by BOTH documents"""
        self.count += 1
        
        # Core data - SHARED EXACTLY between appraisal and engagement letter
        state = random.choice(self.STATES)
        property_address = f"{random.randint(100, 19999)} {fake.street_name()} {random.choice(['St', 'Ave', 'Blvd', 'Dr', 'Ln'])}"
        city = fake.city()
        zip_code = fake.zipcode()
        county = city + ' County'
        
        # People - SAME in both documents
        borrower_name = fake.name()
        lender_name = fake.company() + random.choice([' Mortgage LLC', ' Bank', ' Credit Union'])
        appraiser_name = fake.name()
        appraiser_company = fake.company() + ' Appraisals'
        
        # Property details - SAME in both
        property_type = random.choice(self.PROPERTY_TYPES)
        occupancy = random.choice(self.OCCUPANCY_TYPES)
        report_type = random.choice(self.REPORT_TYPES)
        
        # Dates - logically connected
        order_date = datetime.now() - timedelta(days=random.randint(1, 5))
        inspection_date = order_date + timedelta(days=random.randint(2, 5))
        report_date = inspection_date + timedelta(days=random.randint(3, 7))
        inspection_due = order_date + timedelta(days=random.randint(3, 5))
        report_due = inspection_due + timedelta(days=random.randint(3, 5))
        
        # Property metrics
        year_built = random.randint(1950, 2023)
        gla = random.randint(1200, 5500)
        bedrooms = max(2, min(6, gla // 600))
        bathrooms = round(bedrooms * 0.75 + random.uniform(0, 1), 1)
        
        # Value calculations
        quality = random.choice(['Q1', 'Q2', 'Q3', 'Q4'])
        condition = random.choice(['C1', 'C2', 'C3', 'C4'])
        q_mult = {'Q1': 1.4, 'Q2': 1.2, 'Q3': 1.0, 'Q4': 0.85}[quality]
        c_mult = {'C1': 1.15, 'C2': 1.05, 'C3': 0.95, 'C4': 0.85}[condition]
        appraised_value = int(round(gla * random.randint(180, 450) * q_mult * c_mult, -3))
        
        # Fees
        appraisal_fee = random.choice([475, 500, 525, 550, 575, 600, 650])
        rush_fee = random.choice([0, 75, 100, 150]) if random.random() > 0.7 else 0
        
        # Order identifiers
        order_number = f"ES-{random.randint(100000, 999999)}"
        loan_number = str(random.randint(100000000, 999999999))
        file_no = f"{random.randint(2300, 2500)}-{random.randint(10000, 99999)}"
        parcel_number = f"{random.randint(10,99)}-{random.randint(1000,9999)}-{random.randint(100,999)}"
        license_number = f"R{random.choice(['D','G','A'])}{random.randint(1000, 9999)}"
        
        return {
            # ========== SHARED DATA (EXACT MATCH) ==========
            # Property
            'property_address': property_address,
            'city': city,
            'state': state,
            'zip': zip_code,
            'county': county,
            'property_type': property_type,
            'occupancy': occupancy,
            
            # People
            'borrower_name': borrower_name,
            'lender_name': lender_name,
            'appraiser_name': appraiser_name,
            'appraiser_company': appraiser_company,
            
            # Order info
            'order_number': order_number,
            'loan_number': loan_number,
            'file_no': file_no,
            'report_type': report_type,
            
            # ========== APPRAISAL SPECIFIC ==========
            'parcel_number': parcel_number,
            'license_number': license_number,
            'license_state': state,
            'year_built': str(year_built),
            'quality': quality,
            'condition': condition,
            'bedrooms': str(bedrooms),
            'bathrooms': str(bathrooms),
            'gla': f"{gla:,}",
            'appraised_value': f"{appraised_value:,}",
            'inspection_date': inspection_date.strftime('%m/%d/%Y'),
            'report_date': report_date.strftime('%m/%d/%Y'),
            
            # ========== ENGAGEMENT LETTER SPECIFIC ==========
            'order_date': order_date.strftime('%m/%d/%Y'),
            'inspection_due': inspection_due.strftime('%m/%d/%Y'),
            'report_due': report_due.strftime('%m/%d/%Y'),
            'appraisal_fee': appraisal_fee,
            'rush_fee': rush_fee,
            'client_name': 'Equity Solutions Appraisal Services',
            'client_email': 'orders@equitysolutions.com',
            'client_phone': f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
        }

# Test
gen = UnifiedDataGenerator()
sample = gen.generate()
print("✅ Unified Data Generator ready!")
print(f"   📍 {sample['property_address']}, {sample['city']}, {sample['state']} {sample['zip']}")
print(f"   👤 Borrower: {sample['borrower_name']}")
print(f"   🏦 Lender: {sample['lender_name']}")
print(f"   💰 Value: ${sample['appraised_value']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 3: IMPROVED PDF REGENERATION (Appraisal Reports)
# ═══════════════════════════════════════════════════════════════════
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.colors import white, black

class PDFRegenerator:
    def __init__(self, detector: CompletePDFDetector):
        self.detector = detector
        self.original_data = detector.master_data
    
    def regenerate(self, output_path: str, data: Dict):
        c = canvas.Canvas(output_path, pagesize=letter)
        page_w, page_h = letter
        
        for i, page_img in enumerate(self.detector.page_images):
            img_path = f'/tmp/page_{i}.png'
            page_img.save(img_path, 'PNG', quality=95)
            c.drawImage(img_path, 0, 0, width=page_w, height=page_h, preserveAspectRatio=True)
            
            if i < len(self.detector.page_layouts):
                layout = self.detector.page_layouts[i]
                scale_x = page_w / layout['width']
                scale_y = page_h / layout['height']
                
                for elem in self.detector.elements:
                    if elem.page == i + 1 and elem.element_type == 'text' and elem.is_dynamic:
                        replacement = self._find_replacement(elem.content, data)
                        if replacement:
                            x = elem.x * scale_x
                            y = page_h - (elem.y * scale_y) - (elem.h * scale_y)
                            w = max(elem.w * scale_x, len(str(replacement)) * 5)
                            h = elem.h * scale_y
                            
                            c.setFillColor(white)
                            c.rect(x - 3, y - 3, w + 6, h + 6, fill=True, stroke=False)
                            c.setFillColor(black)
                            c.setFont('Helvetica', min(max(h * 0.75, 7), 10))
                            c.drawString(x, y + (h * 0.25), str(replacement)[:80])
            c.showPage()
        c.save()
        return output_path
    
    def _find_replacement(self, original_text: str, data: Dict):
        if not original_text or len(original_text.strip()) < 2:
            return None
        orig = original_text.lower().strip()
        for field, orig_value in self.original_data.items():
            if orig_value and field in data:
                if str(orig_value).lower().strip() == orig:
                    return str(data[field])
        return None

print("✅ PDF Regenerator ready!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 4: ENGAGEMENT LETTER GENERATOR (Separate PDF, MATCHING data)
# ═══════════════════════════════════════════════════════════════════
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors

class EngagementLetterGenerator:
    """
    Generates corporate-style engagement letters.
    Uses the SAME unified data as the appraisal report.
    All matching fields are IDENTICAL word-for-word.
    """
    
    def __init__(self):
        self.styles = getSampleStyleSheet()
        self._setup_styles()
    
    def _setup_styles(self):
        self.styles.add(ParagraphStyle('Header', parent=self.styles['Heading1'],
            fontSize=14, spaceAfter=6, textColor=colors.black, fontName='Helvetica-Bold'))
        self.styles.add(ParagraphStyle('Section', parent=self.styles['Heading2'],
            fontSize=11, spaceAfter=4, spaceBefore=12, textColor=colors.black, fontName='Helvetica-Bold'))
        self.styles.add(ParagraphStyle('Body', parent=self.styles['Normal'],
            fontSize=10, spaceAfter=6, leading=14, fontName='Helvetica'))
        self.styles.add(ParagraphStyle('BoxText', parent=self.styles['Normal'],
            fontSize=10, fontName='Helvetica-Bold'))
    
    def generate(self, output_path: str, data: Dict):
        """Generate engagement letter using the SAME data as appraisal"""
        doc = SimpleDocTemplate(output_path, pagesize=letter,
            leftMargin=1*inch, rightMargin=1*inch, topMargin=1*inch, bottomMargin=1*inch)
        
        story = []
        
        # 1. HEADER
        story.append(Paragraph(data['client_name'].upper(), self.styles['Header']))
        story.append(Paragraph('Appraisal Engagement / Order Confirmation', self.styles['Section']))
        story.append(Spacer(1, 12))
        
        header_table = Table([
            [f"Order Date: {data['order_date']}", f"Order Number: {data['order_number']}"],
            [f"Loan Number: {data['loan_number']}", '']
        ], colWidths=[3*inch, 3*inch])
        header_table.setStyle(TableStyle([('FONTNAME', (0,0), (-1,-1), 'Helvetica'), ('FONTSIZE', (0,0), (-1,-1), 10)]))
        story.append(header_table)
        story.append(Spacer(1, 18))
        
        # 2. CLIENT INFO
        story.append(Paragraph('Client Information', self.styles['Section']))
        story.append(Paragraph(f"""<b>Client:</b> {data['client_name']}<br/>
        <b>Email:</b> {data['client_email']}<br/>
        <b>Phone:</b> {data['client_phone']}<br/>
        <b>Lender:</b> {data['lender_name']}<br/>
        <b>Borrower:</b> {data['borrower_name']}""", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 3. SUBJECT PROPERTY (BOXED) - EXACT SAME as appraisal
        story.append(Paragraph('Subject Property', self.styles['Section']))
        prop_table = Table([
            [Paragraph(f"<b>{data['property_address']}</b>", self.styles['BoxText'])],
            [Paragraph(f"{data['city']}, {data['state']} {data['zip']}", self.styles['Body'])],
            [Paragraph(f"Property Type: {data['property_type']}", self.styles['Body'])],
            [Paragraph(f"Occupancy: {data['occupancy']}", self.styles['Body'])],
        ], colWidths=[5.5*inch])
        prop_table.setStyle(TableStyle([
            ('BOX', (0,0), (-1,-1), 1, colors.black),
            ('BACKGROUND', (0,0), (-1,-1), colors.white),
            ('TOPPADDING', (0,0), (-1,-1), 6), ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('LEFTPADDING', (0,0), (-1,-1), 10),
        ]))
        story.append(prop_table)
        story.append(Spacer(1, 12))
        
        # 4. PURPOSE
        story.append(Paragraph('Purpose of Appraisal', self.styles['Section']))
        story.append(Paragraph('To develop an opinion of market value.', self.styles['Body']))
        story.append(Paragraph('<b>Intended Use:</b> Mortgage loan underwriting.', self.styles['Body']))
        story.append(Paragraph(f"<b>Intended User:</b> {data['client_name']} and the lender/client identified in this order.", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 5. REPORT TYPE
        story.append(Paragraph('Report Type', self.styles['Section']))
        for t in ['URAR – Form 1004', 'FHA', 'VA', 'Other']:
            check = '☑' if data['report_type'] in t else '☐'
            story.append(Paragraph(f"{check} {t}", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 6. SCOPE OF WORK
        story.append(Paragraph('Scope of Work', self.styles['Section']))
        story.append(Paragraph("""The appraiser shall perform an interior and exterior inspection of the subject property,
        analyze relevant market data, select comparable sales, and prepare an appraisal report
        in compliance with USPAP, Fannie Mae, Freddie Mac, and applicable state regulations.""", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 7. TURN TIME
        story.append(Paragraph('Turn Time', self.styles['Section']))
        story.append(Paragraph(f"<b>Inspection Due:</b> {data['inspection_due']}", self.styles['Body']))
        story.append(Paragraph(f"<b>Report Due:</b> {data['report_due']}", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 8. FEE
        story.append(Paragraph('Fee & Payment', self.styles['Section']))
        fee_text = f"<b>Appraisal Fee:</b> ${data['appraisal_fee']:.2f}"
        if data['rush_fee'] > 0:
            fee_text += f"<br/><b>Rush Fee:</b> ${data['rush_fee']:.2f}"
        fee_text += "<br/><b>Payment Terms:</b> As agreed per vendor agreement."
        story.append(Paragraph(fee_text, self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 9. CONFIDENTIALITY
        story.append(Paragraph('Confidentiality', self.styles['Section']))
        story.append(Paragraph("""This appraisal report is confidential and intended solely for the client and lender
        identified in this order. Distribution to unauthorized parties is prohibited.""", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 10. INDEPENDENCE
        story.append(Paragraph('Independence Certification', self.styles['Section']))
        story.append(Paragraph("""The appraiser certifies that the assignment was accepted and completed without bias,
        coercion, or undue influence, and that compensation is not contingent upon value outcome.""", self.styles['Body']))
        story.append(Spacer(1, 12))
        
        # 11. ACCEPTANCE
        story.append(Paragraph('Acceptance', self.styles['Section']))
        story.append(Paragraph(f"""Acceptance of this appraisal order via the {data['client_name']} portal or email confirmation
        constitutes agreement to all terms and conditions of this engagement.""", self.styles['Body']))
        
        doc.build(story)
        return output_path

print("✅ Engagement Letter Generator ready!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 5: MASS GENERATION (MATCHED PAIRS)
# ═══════════════════════════════════════════════════════════════════
import os
import shutil

NUM_PDFS = 15
OUTPUT_DIR = "generated_documents"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f"{OUTPUT_DIR}/appraisals")
os.makedirs(f"{OUTPUT_DIR}/engagement_letters")

data_gen = UnifiedDataGenerator()
regenerator = PDFRegenerator(detector)
el_generator = EngagementLetterGenerator()

appraisal_files = []
engagement_files = []
all_data = []

print(f"\n{'='*60}")
print(f"🚀 GENERATING {NUM_PDFS} MATCHED DOCUMENT PAIRS")
print(f"   Each pair shares IDENTICAL data (address, borrower, lender, etc.)")
print(f"{'='*60}\n")

for i in range(1, NUM_PDFS + 1):
    # Generate UNIFIED data (used by BOTH documents)
    data = data_gen.generate()
    all_data.append(data)
    
    # Create appraisal PDF
    appraisal_path = f"{OUTPUT_DIR}/appraisals/appraisal_{i:03d}.pdf"
    regenerator.regenerate(appraisal_path, data)
    appraisal_files.append(appraisal_path)
    
    # Create MATCHING engagement letter (SAME data)
    el_path = f"{OUTPUT_DIR}/engagement_letters/engagement_{i:03d}.pdf"
    el_generator.generate(el_path, data)  # Uses SAME data dict
    engagement_files.append(el_path)
    
    print(f"✅ [{i:2d}/{NUM_PDFS}] Matched Pair Generated:")
    print(f"   📍 {data['property_address']}, {data['city']}, {data['state']}")
    print(f"   👤 Borrower: {data['borrower_name']}")
    print(f"   🏦 Lender: {data['lender_name']}")
    print(f"   📋 Order: {data['order_number']} | 💰 Value: ${data['appraised_value']}\n")

print(f"{'='*60}")
print(f"🎉 Generated {len(appraisal_files)} matched pairs!")
print(f"   - Appraisal Reports: {len(appraisal_files)}")
print(f"   - Engagement Letters: {len(engagement_files)}")

In [ ]:
# Create summary CSV showing matched data
import pandas as pd

summary = [{
    'Pair': i+1,
    'Appraisal File': f'appraisal_{i+1:03d}.pdf',
    'Engagement File': f'engagement_{i+1:03d}.pdf',
    'Borrower': d['borrower_name'],
    'Address': d['property_address'],
    'City': d['city'],
    'State': d['state'],
    'Lender': d['lender_name'],
    'Order #': d['order_number'],
    'Value': f"${d['appraised_value']}"
} for i, d in enumerate(all_data)]

df = pd.DataFrame(summary)
df.to_csv(f'{OUTPUT_DIR}/matched_pairs_summary.csv', index=False)
print("📊 Matched Pairs Summary:")
display(df)

In [ ]:
# Download everything
import zipfile

zip_name = "generated_documents.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in appraisal_files:
        zf.write(f, f"appraisals/{os.path.basename(f)}")
    for f in engagement_files:
        zf.write(f, f"engagement_letters/{os.path.basename(f)}")
    zf.write(f'{OUTPUT_DIR}/matched_pairs_summary.csv', 'matched_pairs_summary.csv')

print(f"\n📦 Created {zip_name}")
print(f"   Size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB")
print(f"   Contains: {len(appraisal_files)} appraisals + {len(engagement_files)} engagement letters")
print(f"   ⚠️ All pairs have MATCHING data (address, borrower, lender, etc.)")

files.download(zip_name)
print("\n⬇️ Download started!")